In [1]:
!git clone --branch flow-matching --single-branch https://github.com/Membrizard/ml_conformer_generator.git ml_conformer_generator

Cloning into 'ml_conformer_generator'...
remote: Enumerating objects: 1779, done.
remote: Counting objects: 100% (500/500), done.
remote: Compressing objects: 100% (243/243), done.
remote: Total 1779 (delta 330), reused 314 (delta 248), pack-reused 1279 (from 1)
Receiving objects: 100% (1779/1779), 63.04 MiB | 8.40 MiB/s, done.
Resolving deltas: 100% (1043/1043), done.
Updating files: 100% (210/210), done.


In [2]:
!pip install huggingface_hub
!hf download Membrizard/ml_conformer_generator \
  edm_moi_chembl_15_39.pt \
  --local-dir .

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 798.3/798.3 kB 229.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 522.0 MB/s eta 0:00:00

[notice] A new release of pip is available: 24.2 -> 26.2.1
[notice] To update, run: python -m pip install --upgrade pip
Hint: The `hf-cli` skill is not installed. Run `hf skills add -g --claude` to teach your AI agents how to use the `hf` CLI.

edm_moi_chembl_15_39.pt: downloading bytes: █▋             | 10.7MB,  178kB/s  
edm_moi_chembl_15_39.pt: reconstructing file:   0%| |  172kB / 95.7MB          
edm_moi_chembl_15_39.pt: downloading bytes: ███▍           | 21.9MB, 1.83MB/s  
edm_moi_chembl_15_39.pt: downloading bytes: ███▊           | 24.0MB, 1.92MB/s  
edm_moi_chembl_15_39.pt: downloading bytes: █████          | 32.1MB, 2.36MB/s  
edm_moi_chembl_15_39.pt: downloading bytes: ██████▎        | 40.3MB, 2.93MB/s  
edm_moi_chembl_15_39.pt: reconstructing file:  36%|▎| 34.1MB / 95.7MB, 2.38MB/s
edm_moi_chembl_15_39.pt: downloadin

In [3]:
!pip install "mlconfgen[torch]"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.9/16.9 MB 619.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 38.2/38.2 MB 146.7 MB/s eta 0:00:00a 0:00:01
  Attempting uninstall: numpy
    Found existing installation: numpy 1.26.3
    Uninstalling numpy-1.26.3:
      Successfully uninstalled numpy-1.26.3

[notice] A new release of pip is available: 24.2 -> 26.2.1
[notice] To update, run: python -m pip install --upgrade pip


In [4]:
import json, torch
from pathlib import Path

for name in [
    "moi_distribution_part_1_.json",
    "moi_distribution_part_2.json",
    "moi_distribution_part_3.json",
    "moi_distribution_part_4.json",
]:
    rows = json.load(open(name))
    torch.save(
        {
            "n_atoms": torch.tensor([r["n_atoms"] for r in rows], dtype=torch.int16),
            "context": torch.tensor([r["context"] for r in rows], dtype=torch.float32),
        },
        Path(name).with_suffix(".pt"),
    )
    print(name, len(rows))

moi_distribution_part_1_.json 1641643
moi_distribution_part_2.json 1641643
moi_distribution_part_3.json 1641643
moi_distribution_part_4.json 1641643


In [ ]:
"""
Build a 200k stratified MOI set, then pre-generate teacher (z_T, z_0) pairs.

Writes:
  moi_200k.pt
  teacher_pairs_200k/shardXXXX.pt
"""
from collections import Counter
from datetime import datetime
from pathlib import Path

import shutil
from IPython.display import display, Javascript

import numpy as np
import torch
from tqdm import tqdm

from ml_conformer_generator.src.mlconfgen.egnn import EGNNDynamics
from ml_conformer_generator.src.mlconfgen.equivariant_diffusion import (
    EquivariantDiffusion,
    PredefinedNoiseSchedule,
)
from ml_conformer_generator.src.mlconfgen.utils import CONTEXT_NORMS, MAX_N_NODES
from ml_conformer_generator.src.mlconfgen.utils.mol_utils import prepare_masks

# --------------------- config ---------------------
device = "cuda"
SEED = 43
TARGET = 200_000
N_Q = 10                 # Izz quantile bins per n_atoms
TEACHER_STEPS = 100
BATCH = 1024
PAD_TO = MAX_N_NODES     # 39
SHARD_SIZE = 10_000

EDM_WEIGHTS = "edm_moi_chembl_15_39.pt"
MOI_SOURCES = [
    "moi_distribution_part_1_.pt",   # prefer .pt; .json also works
    "moi_distribution_part_2.pt",
    "moi_distribution_part_3.pt",
    "moi_distribution_part_4.pt",
]
OUT_MOI = Path("moi_200k.pt")
OUT_DIR = Path("part_2_teacher_pairs_200k")
OUT_DIR.mkdir(parents=True, exist_ok=True)
LOG_PATH = OUT_DIR / "generate.log"

# --------------------- helpers ---------------------
def log(msg: str) -> None:
    line = f"{datetime.now().isoformat(timespec='seconds')}  {msg}"
    print(line)
    with open(LOG_PATH, "a") as f:
        f.write(line + "\n")


def load_pool(paths):
    """Concat n_atoms (int16) + context (float32, 3) from .pt or .json."""
    n_list, c_list = [], []
    for p in paths:
        p = Path(p)
        json_p = p.with_suffix(".json") if p.suffix == ".pt" else p
        pt_p = p.with_suffix(".pt") if p.suffix != ".pt" else p
        if pt_p.exists():
            pack = torch.load(pt_p, map_location="cpu")
            n_list.append(pack["n_atoms"].to(torch.int16))
            c_list.append(pack["context"].float())
            log(f"pool {pt_p.name}  n={n_list[-1].shape[0]}")
        elif json_p.exists():
            import json
            rows = json.load(open(json_p))
            n_list.append(torch.tensor([r["n_atoms"] for r in rows], dtype=torch.int16))
            c_list.append(torch.tensor([r["context"] for r in rows], dtype=torch.float32))
            log(f"pool {json_p.name}  n={n_list[-1].shape[0]}")
        else:
            log(f"skip missing {p}")
    if not n_list:
        raise FileNotFoundError("no MOI source files found")
    return torch.cat(n_list), torch.cat(c_list)


def stratified_indices(n_atoms: np.ndarray, context: np.ndarray, target: int, n_q: int, seed: int):
    """~equal n_atoms; within each size, equal draws from Izz quantile bins."""
    rng = np.random.default_rng(seed)
    sizes = np.arange(int(n_atoms.min()), int(n_atoms.max()) + 1)
    by_size = {int(n): np.flatnonzero(n_atoms == n) for n in sizes}
    avail = {n: len(ix) for n, ix in by_size.items()}

    per = target // len(sizes)
    alloc = {n: min(per, avail[n]) for n in sizes}
    short = target - sum(alloc.values())
    for n in sorted(sizes, key=lambda n: avail[n] - alloc[n], reverse=True):
        if short <= 0:
            break
        take = min(short, avail[n] - alloc[n])
        alloc[n] += take
        short -= take

    picked = []
    for n, k in alloc.items():
        idx = by_size[n]
        if k == 0:
            continue
        if len(idx) <= k:
            picked.append(idx)
            continue
        izz = context[idx, 2]
        edges = np.quantile(izz, np.linspace(0.0, 1.0, n_q + 1))
        edges[0] -= 1e-6
        edges[-1] += 1e-6
        # unique edges (flat bins happen)
        edges = np.unique(edges)
        bins = np.clip(np.searchsorted(edges, izz, side="right") - 1, 0, len(edges) - 2)
        n_bins = bins.max() + 1
        want = np.full(n_bins, k // n_bins)
        want[: k % n_bins] += 1

        chosen, unused, leftover = [], [], 0
        for b in range(n_bins):
            in_b = idx[bins == b]
            w = int(want[b]) if b < len(want) else 0
            if len(in_b) <= w:
                chosen.append(in_b)
                leftover += w - len(in_b)
            else:
                sel = rng.choice(in_b, w, replace=False)
                chosen.append(sel)
                unused.append(np.setdiff1d(in_b, sel))
        if leftover:
            pool = np.concatenate(unused) if unused else np.array([], dtype=idx.dtype)
            if len(pool) >= leftover:
                chosen.append(rng.choice(pool, leftover, replace=False))
            elif len(pool):
                chosen.append(pool)
        picked.append(np.concatenate(chosen))

    out = np.concatenate(picked)
    rng.shuffle(out)  # mix sizes across shards
    return out[:target], alloc


def make_batch(n_atoms, context, norms, device, pad_to=PAD_TO):
    n_atoms = n_atoms.to(device=device, dtype=torch.long)
    ctx = context.to(device=device, dtype=torch.float32)
    node_mask, edge_mask = prepare_masks(n_atoms, pad_to, device)
    normed = (ctx - norms["mean"]) / norms["mad"]
    batch_context = normed.unsqueeze(1).expand(-1, pad_to, -1) * node_mask
    return node_mask, edge_mask, batch_context


def flush_shard(buf, shard_i):
    n = sum(t.shape[0] for t in buf["n_atoms"])
    path = OUT_DIR / f"shard{shard_i:04d}.pt"
    torch.save(
        {
            "z_T": torch.cat(buf["z_T"]).half().contiguous(),
            "x1": torch.cat(buf["x1"]).half().contiguous(),
            "n_atoms": torch.cat(buf["n_atoms"]).to(torch.int16),
            "context": torch.cat(buf["context"]).contiguous(),
            "teacher_steps": TEACHER_STEPS,
            "pad_to": PAD_TO,
        },
        path,
    )
    log(f"wrote {path.name}  n={n}")
    return path


# --------------------- stage 1: 200k MOI ---------------------
n_all, ctx_all = load_pool(MOI_SOURCES)
log(f"pool total {n_all.shape[0]}")

idx, alloc = stratified_indices(
    n_all.numpy(), ctx_all.numpy(), target=TARGET, n_q=N_Q, seed=SEED
)
n_200k = n_all[idx].contiguous()
ctx_200k = ctx_all[idx].contiguous()
torch.save({"n_atoms": n_200k, "context": ctx_200k, "seed": SEED, "index": torch.from_numpy(idx)}, OUT_MOI)

counts = Counter(n_200k.tolist())
log(f"wrote {OUT_MOI}  n={n_200k.shape[0]}")
log("n_atoms alloc: " + " ".join(f"{n}:{counts.get(n, 0)}" for n in range(15, 40)))
log(
    "Izz q05/50/95  pool="
    f"{ctx_all[:, 2].quantile(0.05):.1f}/{ctx_all[:, 2].quantile(0.5):.1f}/{ctx_all[:, 2].quantile(0.95):.1f}"
    "  200k="
    f"{ctx_200k[:, 2].quantile(0.05):.1f}/{ctx_200k[:, 2].quantile(0.5):.1f}/{ctx_200k[:, 2].quantile(0.95):.1f}"
)

# --------------------- stage 2: teacher pairs ---------------------
dyn = EGNNDynamics(in_node_nf=9, context_node_nf=3, hidden_nf=420, device=device)
teacher = EquivariantDiffusion(
    dynamics=dyn, in_node_nf=8, timesteps=1000, noise_precision=1e-5
)
edm_ckpt = torch.load(EDM_WEIGHTS, map_location=device)
teacher.load_state_dict(edm_ckpt["state_dict"])
teacher.gamma = PredefinedNoiseSchedule(timesteps=TEACHER_STEPS, precision=1e-5)
teacher.time_steps = torch.flip(torch.arange(TEACHER_STEPS, device=device), [0])
teacher.T = TEACHER_STEPS
teacher.to(device).eval()
for p in teacher.parameters():
    p.requires_grad_(False)

norms = {
    k: torch.tensor(v, device=device, dtype=torch.float32)
    for k, v in edm_ckpt.get("context_norms", CONTEXT_NORMS).items()
}

n_samples = n_200k.shape[0]
shard_i = 0
while (OUT_DIR / f"shard{shard_i:04d}.pt").exists():
    shard_i += 1
start = shard_i * SHARD_SIZE
log(f"pairs resume at sample {start} shard {shard_i} / {n_samples}")

buf = {"z_T": [], "x1": [], "n_atoms": [], "context": []}
buf_n = 0
pbar = tqdm(range(start, n_samples - BATCH + 1, BATCH), desc="pairs")
for i in pbar:
    na = n_200k[i : i + BATCH]
    ctx = ctx_200k[i : i + BATCH]
    node_mask, edge_mask, context = make_batch(na, ctx, norms, device)
    B, N, _ = node_mask.shape
    with torch.inference_mode():
        z_T = teacher.sample_combined_position_feature_noise(B, N, node_mask)
        x1 = teacher.teach(z_T, node_mask, edge_mask, context)
    buf["z_T"].append(z_T.cpu())
    buf["x1"].append(x1.cpu())
    buf["n_atoms"].append(na.cpu())
    buf["context"].append(ctx.cpu())
    buf_n += B
    if buf_n >= SHARD_SIZE:
        flush_shard(buf, shard_i)
        buf = {"z_T": [], "x1": [], "n_atoms": [], "context": []}
        buf_n = 0
        shard_i += 1

if buf_n:
    flush_shard(buf, shard_i)
log("done")

# --- export to local ---

# Create teacher pairs archive
pairs_dir = Path("teacher_pairs_200k")
pairs_zip = Path("teacher_pairs_200k.zip")

if pairs_zip.exists():
    pairs_zip.unlink()

shutil.make_archive(
    "teacher_pairs_200k",
    "zip",
    root_dir=pairs_dir
)

# Put everything into one export ZIP
export_dir = Path("_export")
export_dir.mkdir(exist_ok=True)

shutil.copy2("moi_200k.pt", export_dir / "moi_200k.pt")
shutil.copy2(pairs_zip, export_dir / pairs_zip.name)

final_zip = Path("mlconfgen_pairs_200k.zip")

if final_zip.exists():
    final_zip.unlink()

shutil.make_archive(
    "mlconfgen_pairs_200k",
    "zip",
    root_dir=export_dir
)

print(
    f"Export ready: {final_zip} "
    f"({final_zip.stat().st_size / 1e6:.1f} MB)"
)

# Force browser download
display(Javascript(f"""
    (() => {{
        const base = window.location.pathname.split('/lab')[0];
        const a = document.createElement('a');
        a.href = base + '/files/{final_zip.name}';
        a.download = '{final_zip.name}';
        document.body.appendChild(a);
        a.click();
        a.remove();
    }})();
"""))


/tmp/ipykernel_339/1972253966.py:65: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  pack = torch.load(pt_p, map_location="cpu")


2026-09-11T19:56:12  pool moi_distribution_part_1_.pt  n=1641643
2026-09-11T19:56:12  pool moi_distribution_part_2.pt  n=1641643
2026-09-11T19:56:12  pool moi_distribution_part_3.pt  n=1641643
2026-09-11T19:56:12  pool moi_distribution_part_4.pt  n=1641643
2026-09-11T19:56:12  pool total 6566572
2026-09-11T19:56:15  wrote moi_200k.pt  n=200000
2026-09-11T19:56:15  n_atoms alloc: 15:8000 16:8000 17:8000 18:8000 19:8000 20:8000 21:8000 22:8000 23:8000 24:8000 25:8000 26:8000 27:8000 28:8000 29:8000 30:8000 31:8000 32:8000 33:8000 34:8000 35:8000 36:8000 37:8000 38:8000 39:8000
2026-09-11T19:56:18  Izz q05/50/95  pool=167.2/477.7/1104.8  200k=131.4/477.8/1216.6


/tmp/ipykernel_339/1972253966.py:195: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  edm_ckpt = torch.load(EDM_WEIGHTS, map_location=device)


2026-09-11T19:56:19  pairs resume at sample 0 shard 0 / 200000


pairs:   5%|▌         | 10/195 [1:32:41<28:34:44, 556.14s/it]

...

pairs:  21%|██        | 40/195 [6:10:47<23:56:16, 555.98s/it]

2026-09-12T02:07:06  wrote shard0003.pt  n=10240


pairs:  26%|██▌       | 50/195 [7:43:27<22:23:56, 556.11s/it]

2026-09-12T03:39:46  wrote shard0004.pt  n=10240


pairs:  30%|███       | 59/195 [9:06:51<21:00:14, 555.99s/it]